<a href="https://colab.research.google.com/github/mjh5153/comply-api-blueprint/blob/comply-ml-train/0_BERT_Complete_Explanatory_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BERT (Bidirectional Encoder Representations from Transformers)
# Complete End-to-End Hands-on Lab

## Learning Objectives
- Understand why BERT was developed
- Learn the Transformer Encoder architecture
- Understand tokenization, embeddings, self-attention, and masked language modeling
- Use Hugging Face Transformers
- Perform sentiment analysis
- Generate embeddings
- Fine-tune BERT for text classification
- Practice exercises and mini project



# 1. Why BERT?

Older models (RNN, LSTM, Word2Vec) struggled to understand context from both directions.

Sentence:
> I went to the bank to deposit money.

Sentence:
> We sat near the river bank.

The word **bank** has different meanings.

BERT reads words **bidirectionally**, allowing it to understand context from both the left and right sides of a token.


# 2. What does BERT stand for?

**B**idirectional  
**E**ncoder  
**R**epresentations from  
**T**ransformers

BERT uses only the **Transformer Encoder** stack.


# 3. BERT Architecture

Pipeline:

Input Text
→ Tokenizer
→ Token Embeddings
→ Positional Embeddings
→ Segment Embeddings
→ Transformer Encoder Layers
→ Contextual Embeddings
→ Prediction Head

Key concepts:
- WordPiece Tokenization
- Self-Attention
- Multi-Head Attention
- Feed Forward Network
- Layer Normalization
- Residual Connections


# 4. Installation

```bash
pip install transformers datasets torch
```


In [4]:
pip install transformers datasets torch

In [1]:
from transformers import AutoTokenizer, AutoModel
import torch

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

print("BERT loaded successfully")


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT loaded successfully


## 5. Tokenization

In [5]:
text = "Transformers changed modern NLP."

tokens = tokenizer.tokenize(text)
ids = tokenizer.convert_tokens_to_ids(tokens)

print("Tokens:", tokens)
print("Token IDs:", ids)


Tokens: ['transformers', 'changed', 'modern', 'nl', '##p', '.']
Token IDs: [19081, 2904, 2715, 17953, 2361, 1012]


Notice special tokens:

- `[CLS]` : classification token
- `[SEP]` : sentence separator
- `[PAD]` : padding
- `[MASK]` : masked token used during pretraining


In [7]:
encoded = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**encoded)

print("Last Hidden State Shape:", outputs.last_hidden_state.shape)
print("CLS Embedding Shape:", outputs.last_hidden_state[:,0,:].shape)


Last Hidden State Shape: torch.Size([1, 8, 768])
CLS Embedding Shape: torch.Size([1, 768])


## 6. Sentence Embeddings

The embedding of the `[CLS]` token is commonly used as a sentence representation for many downstream tasks.


In [ ]:
sentence_embedding = outputs.last_hidden_state[:,0,:]
print(sentence_embedding)


tensor([[-8.8557e-01, -2.3598e-01,  3.0514e-01,  5.1139e-02, -3.1519e-01,
         -3.4072e-01,  8.9488e-02,  2.9115e-01, -3.2803e-01, -1.1378e-01,
          2.4494e-01, -9.3118e-02, -2.2243e-01,  4.7522e-01, -9.4131e-02,
          1.9778e-02, -3.9051e-01, -2.6242e-02, -4.6914e-03, -1.3121e-01,
          1.6702e-01, -2.6573e-01,  5.4383e-02, -2.6104e-01, -4.2058e-01,
          9.0368e-02,  2.0926e-02, -3.1056e-01, -6.5450e-02,  3.7850e-01,
         -2.5972e-01,  1.8261e-01, -5.1468e-01, -1.3345e-01,  3.5005e-01,
         -5.4473e-01, -1.9303e-01, -3.0933e-01,  1.0516e-01, -9.5251e-02,
         -2.2204e-01, -1.3048e-01,  3.0953e-01,  5.0793e-02, -4.7094e-01,
          1.3063e-01, -2.7474e+00,  1.5293e-01, -7.7470e-01, -6.2154e-01,
         -8.9829e-02, -2.8426e-01,  4.1947e-01,  5.7559e-01, -5.5904e-01,
          6.6542e-01,  6.6682e-02,  3.8260e-01,  3.7154e-01,  2.2435e-01,
          2.3460e-01,  1.2614e-01, -3.1225e-01, -9.5196e-02,  3.8176e-02,
          8.8480e-02, -2.8143e-01,  1.

# 7. Masked Language Modeling

During pretraining, BERT hides words and learns to predict them.

Example:

The capital of France is **[MASK]**

Expected prediction: **Paris**


In [9]:
from transformers import pipeline

fill_mask = pipeline("fill-mask", model="bert-base-uncased")

fill_mask("The capital of France is [MASK].")


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'score': 0.41678905487060547,
  'token': 3000,
  'token_str': 'paris',
  'sequence': 'the capital of france is paris.'},
 {'score': 0.07141662389039993,
  'token': 22479,
  'token_str': 'lille',
  'sequence': 'the capital of france is lille.'},
 {'score': 0.06339248269796371,
  'token': 10241,
  'token_str': 'lyon',
  'sequence': 'the capital of france is lyon.'},
 {'score': 0.044447533786296844,
  'token': 16766,
  'token_str': 'marseille',
  'sequence': 'the capital of france is marseille.'},
 {'score': 0.03029720112681389,
  'token': 7562,
  'token_str': 'tours',
  'sequence': 'the capital of france is tours.'}]

# 8. Sentiment Analysis with BERT

In [10]:
classifier = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

classifier([
    "This course is amazing.",
    "The movie was boring."
])


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

[{'label': 'POSITIVE', 'score': 0.9998829364776611},
 {'label': 'NEGATIVE', 'score': 0.9997908473014832}]

# 9. Fine-Tuning Workflow

1. Collect dataset
2. Tokenize text
3. Create train/test split
4. Load `AutoModelForSequenceClassification`
5. Train with Trainer API
6. Evaluate accuracy/F1
7. Save the model

Fine-tuning updates pretrained BERT weights for your specific task.


In [ ]:
from transformers import AutoModelForSequenceClassification

classification_model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

print(classification_model.config)


Loading weights: 100%|████████████████████| 199/199 [00:00<00:00, 11723.01it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from t

BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "dtype": "float32",
  "eos_token_id": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "tie_word_embeddings": true,
  "transformers_version": "5.13.0",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}



In [3]:
# Parsing -
# Rule based ( traditional )
# -> tnlp,
# top down parsing -> TNLP,
# bottom up parsing -> tnlp,
# statical dependency parsing ( bag of words, ML) -> snlp,
# neuralDependencey-> modern NLP

# THE BOY KICKED THE FOOTBALL
# Wh, what, receiver

# Types of parsing
# 1. Consitutency, parsing -> Tree representation of grammatical structure of sentence,
# 2. Dependency parsing,

# Ex: The boy eats an apple
  # Dependency - which word diepends on which other word in sentence
  ## The boy kicked the football

  # Relation Extraction


# 10. Real-World Applications

- Sentiment Analysis
- Named Entity Recognition
- Question Answering
- Text Classification
- Semantic Search
- Chatbots
- Resume Screening
- Healthcare NLP
- Legal Document Analysis


# 11. Advantages

- Bidirectional context
- Pretrained on massive corpora
- High accuracy
- Transfer learning
- Strong performance across many NLP tasks


# 12. Limitations

- Computationally expensive
- Input length limit (typically 512 tokens)
- Large memory usage
- Slower than lightweight models


# 13. Practice Exercises

1. Tokenize five different sentences.
2. Use the fill-mask pipeline on your own examples.
3. Compare BERT embeddings for:
   - Apple released a phone.
   - I ate an apple.
4. Perform sentiment analysis on product reviews.
5. Fine-tune BERT on a custom text classification dataset.


# 14. Mini Project

Build a Product Review Analyzer.

Requirements:
- Load customer reviews
- Tokenize using BERT
- Perform sentiment analysis
- Store positive/negative predictions
- Visualize sentiment distribution


## Mini Assignment Solution

This section addresses the NLP parsing task using `spaCy` to identify grammatical components and visualize dependencies.

In [11]:
# 1. Setup and Model Download
import spacy
from spacy import displacy

# Download the small English model
!python -m spacy download en_core_web_sm

nlp = spacy.load("en_core_web_sm")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 25.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [12]:
# 2, 3, & 4. Extraction and Visualization

def extract_details(doc):
    results = {
        "Subject": [],
        "Verb": [],
        "Object": [],
        "Modifiers": [],
        "Locations": []
    }

    for token in doc:
        # Identify Verb
        if token.pos_ == "VERB" or token.dep_ == "ROOT":
            results["Verb"].append(token.text)

        # Identify Subjects
        if "subj" in token.dep_:
            results["Subject"].append(token.text)

        # Identify Objects
        if "obj" in token.dep_ and token.dep_ != "pobj":
            results["Object"].append(token.text)

        # Identify Modifiers (Adjectives, Adverbs, Determiners)
        if token.dep_ in ["amod", "advmod", "det", "compound"]:
            results["Modifiers"].append(f"{token.text} -> {token.head.text}")

        # Identify Locations (prep + pobj)
        if token.dep_ == "pobj":
            results["Locations"].append(f"{token.head.text} {token.text}")

    return results

sentences = [
    "Rahul lives in Delhi.",
    "Microsoft acquired GitHub.",
    "The quick brown fox jumps over the lazy dog."
]

for sentence in sentences:
    doc = nlp(sentence)
    details = extract_details(doc)

    print(f"\nSentence: {sentence}")
    for key, value in details.items():
        print(f"{key}: {', '.join(value) if value else 'None'}")

    # Visualize dependency graph
    displacy.render(doc, style="dep", jupyter=True, options={'distance': 100})


Sentence: Rahul lives in Delhi.
Subject: Rahul
Verb: lives
Object: None
Modifiers: None
Locations: in Delhi



Sentence: Microsoft acquired GitHub.
Subject: Microsoft
Verb: acquired
Object: GitHub
Modifiers: None
Locations: None



Sentence: The quick brown fox jumps over the lazy dog.
Subject: fox
Verb: jumps
Object: None
Modifiers: The -> fox, quick -> fox, brown -> fox, the -> dog, lazy -> dog
Locations: over dog


### Combined Solution (Single Cell)

In [13]:
# Combined Solution: Parsing, SVO Extraction, and Visualization
import spacy
from spacy import displacy

# 1. Download and load the model
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    !python -m spacy download en_core_web_sm
    nlp = spacy.load("en_core_web_sm")

# 2. Define the extraction logic
def extract_details(doc):
    results = {
        "Subject": [],
        "Verb": [],
        "Object": [],
        "Modifiers": [],
        "Locations": []
    }
    for token in doc:
        # Verb / Root
        if token.pos_ == "VERB" or token.dep_ == "ROOT":
            results["Verb"].append(token.text)
        # Subjects
        if "subj" in token.dep_:
            results["Subject"].append(token.text)
        # Objects (excluding prepositional objects for the Object category)
        if "obj" in token.dep_ and token.dep_ != "pobj":
            results["Object"].append(token.text)
        # Modifiers
        if token.dep_ in ["amod", "advmod", "det", "compound"]:
            results["Modifiers"].append(f"{token.text} -> {token.head.text}")
        # Locations (Preposition + Object of Preposition)
        if token.dep_ == "pobj":
            results["Locations"].append(f"{token.head.text} {token.text}")
    return results

# 3. Process sentences
sentences = [
    "Rahul lives in Delhi.",
    "Microsoft acquired GitHub.",
    "The quick brown fox jumps over the lazy dog."
]

for sentence in sentences:
    doc = nlp(sentence)
    details = extract_details(doc)

    print(f"\nSentence: {sentence}")
    for key, value in details.items():
        print(f"{key}: {', '.join(value) if value else 'None'}")

    # 4. Visualize dependency graph
    displacy.render(doc, style="dep", jupyter=True, options={'distance': 100})


Sentence: Rahul lives in Delhi.
Subject: Rahul
Verb: lives
Object: None
Modifiers: None
Locations: in Delhi



Sentence: Microsoft acquired GitHub.
Subject: Microsoft
Verb: acquired
Object: GitHub
Modifiers: None
Locations: None



Sentence: The quick brown fox jumps over the lazy dog.
Subject: fox
Verb: jumps
Object: None
Modifiers: The -> fox, quick -> fox, brown -> fox, the -> dog, lazy -> dog
Locations: over dog


# Learning Outcomes

After completing this lab, you will be able to:
- Explain BERT architecture
- Understand tokenization and embeddings
- Explain masked language modeling
- Use Hugging Face Transformers
- Generate contextual embeddings
- Perform sentiment analysis
- Understand the workflow for fine-tuning BERT
